# Data Rescue Kit
A standalone reference: choose the symptom below. Each repair has a small diagnosis, a fix, and a check. Run the setup once, then run the code for your chosen symptom. Examples are invented. **Do not apply a repair until you can explain what it changes.**

[View on GitHub](https://github.com/mathguy-uf/MAD2502/blob/main/applied_data/04_data_rescue_kit.ipynb) · [Run in Google Colab](https://colab.research.google.com/github/mathguy-uf/MAD2502/blob/main/applied_data/04_data_rescue_kit.ipynb) · [Launch Binder/JupyterLab](https://mybinder.org/v2/gh/mathguy-uf/MAD2502/main?urlpath=lab/tree/applied_data/04_data_rescue_kit.ipynb)

Run cells from the top with **Shift+Enter**. Restart the kernel and run all to check your work. Each tutorial includes its own imports and data. Hosted sessions are temporary: download your work before leaving.

In [ ]:
# If an import fails, uncomment and run this once, then restart the kernel.
# %pip install pandas

In [ ]:
from pathlib import Path
from io import StringIO
import pandas as pd
from IPython.display import display

## FileNotFoundError / wrong working directory
A relative path starts in the kernel's current folder, which may differ from the notebook folder. Inspect that folder; check spelling and capitalization. This expected error is caught so the lesson can continue.

**Predict:** what should the diagnostic show before the fix?

In [ ]:
print("Working folder:", Path.cwd())
path = Path("missing_example.csv")
try:
    pd.read_csv(path)
except FileNotFoundError:
    print("Expected error: this demonstration file does not exist.")
# For your file, choose a real path from your Files pane.
# This tiny in-memory CSV shows what a successful read looks like:
fixed = pd.read_csv(StringIO("item,value\nA,3\nB,5\n"))
display(fixed.head())

## Numbers stored as strings; dollars and commas
Inspect the type before computing. Preserve the raw column. Coercion marks invalid entries missing; check which entries became missing.

**Predict:** what should the diagnostic show before the fix?

In [ ]:
money = pd.DataFrame({"raw": ["$1,200", "$0", "unknown", None]})
text = money["raw"].astype("string").str.replace("$", "", regex=False).str.replace(",", "", regex=False)
money["value"] = pd.to_numeric(text, errors="coerce")
display(money)
print(money.dtypes)
display(money[money["raw"].notna() & money["value"].isna()])

## Unexpected NaN
NaN is a missing value. Missing can mean unrecorded or unconvertible, not zero. Choose a policy for the particular analysis and report its denominator.

**Predict:** what should the diagnostic show before the fix?

In [ ]:
scores = pd.DataFrame({"value": [4, None, 8]})
print("Missing:", scores["value"].isna().sum())
print("Mean of known values:", scores["value"].mean())
print("Known values used:", scores["value"].count(), "of", len(scores))

## Duplicate rows
Equal measurements can be legitimate repeated observations. Remove duplicates only when the record identity or source confirms duplication. Here repeated event_id means the same imported event.

**Predict:** what should the diagnostic show before the fix?

In [ ]:
events = pd.DataFrame({"event_id": [1, 2, 2], "attendance": [10, 20, 20]})
display(events[events.duplicated(keep=False)])
unique_events = events.drop_duplicates(subset=["event_id"])
print("Before / after:", len(events), len(unique_events))

## Strange column names / KeyError
Spaces and capitalization are part of a name. Normalize them carefully and check that two original names did not collapse to the same name.

**Predict:** what should the diagnostic show before the fix?

In [ ]:
table = pd.DataFrame({" Event Name ": ["A"], " Cost ": [10]})
print(list(table.columns))
table.columns = table.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
assert table.columns.is_unique, "Renaming created duplicate column labels."
display(table[["event_name", "cost"]])

## Dates stored as text
Use an explicit format when you know it. Mixed formats need deliberate parsing; a guessed month/day order may silently change meaning. These two documented formats are ISO and US month/day/year.

**Predict:** what should the diagnostic show before the fix?

In [ ]:
dates = pd.Series(["2026-10-01", "10/05/2026", "bad date"])
iso = pd.to_datetime(dates, format="%Y-%m-%d", errors="coerce")
us = pd.to_datetime(dates, format="%m/%d/%Y", errors="coerce")
parsed = iso.fillna(us)
display(pd.DataFrame({"raw": dates, "parsed": parsed}))
print("Unparsed:", dates[parsed.isna()].tolist())

## Nested JSON / wrong object type
A list supports positions; a dictionary supports keys; a DataFrame supports column labels. Inspect type and one small record. json_normalize flattens nested dictionaries.

**Predict:** what should the diagnostic show before the fix?

In [ ]:
payload = {"results": [{"name": "A", "location": {"city": "Gainesville"}}]}
print(type(payload), list(payload))
records = payload["results"]
print(type(records), records[0])
flat = pd.json_normalize(records)
display(flat)

## Empty API response / empty DataFrame
An empty list can be a successful request with no matches. Check HTTP status first in live code, then the query and time range. Do not invent observations. Preserve the expected columns.

**Predict:** what should the diagnostic show before the fix?

In [ ]:
records = []
empty = pd.DataFrame(records, columns=["speciesCode", "howMany"])
print("Empty:", empty.empty, "shape:", empty.shape)
if empty.empty:
    print("No records: check the query before computing a summary.")
values = pd.DataFrame({"cost": [0, 10, 20]})
selected = values[values["cost"] > 5000]
print("Filter returned", len(selected), "rows; actual range:", values["cost"].min(), values["cost"].max())

## Combining tables / mismatched columns
concat stacks rows and aligns by column name. Rename matching concepts first; do not equate columns with different units. A missing unmatched field is not automatically zero.

**Predict:** what should the diagnostic show before the fix?

In [ ]:
first = pd.DataFrame({"event": ["A"], "cost_usd": [10]})
second = pd.DataFrame({"event": ["B"], "price_usd": [15]})
print("Columns before:", list(first.columns), list(second.columns))
second = second.rename(columns={"price_usd": "cost_usd"})
combined = pd.concat([first, second], ignore_index=True)
display(combined)
assert len(combined) == len(first) + len(second)

## Combining tables by a key
merge joins columns using an identifier. Duplicate keys can multiply rows. validate checks the intended relationship; indicator shows unmatched keys. Here each site should occur once in the lookup table.

**Predict:** what should the diagnostic show before the fix?

In [ ]:
observations = pd.DataFrame({"site_id": [1, 1, 2], "count": [3, 4, 5]})
sites = pd.DataFrame({"site_id": [1, 2], "site": ["Garden", "Lake"]})
joined = observations.merge(sites, on="site_id", how="left", validate="many_to_one", indicator=True)
display(joined)
print("Unmatched rows:", (joined["_merge"] != "both").sum())

## Practice and connect
Change one input in a repair, predict the result, and rerun. Try an unparseable dollar amount, a repeated join key, or a filter threshold of 10. Explain why deleting all missing rows is not a universal repair.

For a project: record the symptom, diagnosis, change, row counts before/after, and reason in your journal. Restart and run all before trusting a final result.